0. 블록 분할

In [2]:
import numpy as np
import pandas as pd

N_ORIG = 7377418

BLOCKS = {
    "train_a": (0.2, 0.4),
    "train_b": (0.4, 0.6),
    "valid": (0.6, 0.8),
    "test": (0.8, 1.0),
}
LABEL_BOUND = {"train_a": 0.0, "train_b": 0.0, "valid": 0.045, "test": 0.37}

df = pd.read_csv("data.csv", engine="pyarrow")
df = df.sort_values("row_order", kind="stable").reset_index(drop=True)
ro = df["row_order"].to_numpy()

def cut(frac):
    return int(np.searchsorted(ro, frac * N_ORIG, side="left"))

prep = df.iloc[:cut(0.2)]
blocks = {name: df.iloc[cut(lo):cut(hi)] for name, (lo, hi) in BLOCKS.items()}

def history(name):
    T = BLOCKS[name][0]
    count_hist = df.iloc[:cut(T)]
    g0_hist = df.iloc[:cut(T)]
    g30_hist = df.iloc[:cut(LABEL_BOUND[name])]
    return count_hist, g0_hist, g30_hist

train_a = blocks["train_a"]
train_b = blocks["train_b"]
valid = blocks["valid"]
test = blocks["test"]

prep.to_parquet("data_prep.parquet", index=False)
for name, b in blocks.items():
    b.to_parquet(f"data_{name}.parquet", index=False)

rows = []
for name, b in blocks.items():
    c, g0, g30 = history(name)
    rows.append({
        "block": name,
        "행 수": len(b),
        "이용자 수": b["msno"].nunique(),
        "기록(count) 행 수": len(c),
        "라벨 기록 행 수(30일)": len(g30),
        "라벨 기록 행 수(지연 없음)": len(g0),
        "target 평균": round(b["target"].mean(), 4),
    })

print("준비 기간(기록 전용) 행 수:", len(prep))
print(pd.DataFrame(rows).to_string(index=False))
print("저장 완료: data_prep / data_train_a / data_train_b / data_valid / data_test (.parquet)")

준비 기간(기록 전용) 행 수: 1469890
  block     행 수  이용자 수  기록(count) 행 수  라벨 기록 행 수(30일)  라벨 기록 행 수(지연 없음)  target 평균
train_a 1470413  20260        1469890               0           1469890     0.5483
train_b 1469956  21251        2940303               0           2940303     0.4699
  valid 1470833  21858        4410259          330703           4410259     0.4277
   test 1471022  22616        5881092         2719739           5881092     0.4045
저장 완료: data_prep / data_train_a / data_train_b / data_valid / data_test (.parquet)


1. 합격선과 평가 기준 확정

In [3]:
import json
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.calibration import calibration_curve

CRITERIA = {
    "작성시각": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "비교 대상": "인기도 기준 모델 (블록 시작 시점 T 이전 곡 청취 수)",
    "평가 구간": "valid에서 모델 선택, test에서 최종 1회 확인",
    "주 지표": "AUC",
    "보조 지표": ["로그손실", "보정 곡선(10분위)", "ECE"],
    "합격선": "모델 AUC > 인기도 AUC 이고, 사용자 단위 부트스트랩 AUC 차이의 95% 신뢰구간 하한 > 0",
    "부트스트랩": {"단위": "사용자(msno) 복원추출", "반복수": 300, "신뢰수준": 0.95, "seed": 42},
    "미충족 시": "특성, 누수, 분포 변화를 점검해 개선한 뒤 valid에서 다시 확인",
    "금지": "test 결과를 보고 모델, 특성, 합격선을 바꾸지 않는다",
}

if not os.path.exists("합격선_기준.json"):
    with open("합격선_기준.json", "w", encoding="utf-8") as f:
        json.dump(CRITERIA, f, ensure_ascii=False, indent=2)
    with open("합격선_기준.md", "w", encoding="utf-8") as f:
        f.write("# 모델 A 합격선과 평가 기준 (학습 전 확정)\n\n")
        for k, v in CRITERIA.items():
            f.write(f"- **{k}**: {v}\n")
    print("합격선 기준 저장 완료")
else:
    print("이미 저장된 합격선 기준이 있어 덮어쓰지 않았습니다")

def popularity_score(block_df, history_df):
    cnt = history_df["song_id"].value_counts()
    return block_df["song_id"].map(cnt).fillna(0).to_numpy(dtype=float)

def expected_calibration_error(y, p, n_bins=10):
    bins = pd.qcut(p, q=n_bins, duplicates="drop")
    g = pd.DataFrame({"y": y, "p": p, "b": bins}).groupby("b", observed=True)
    s = g.agg(y=("y", "mean"), p=("p", "mean"), n=("y", "size"))
    return float((s["n"] * (s["y"] - s["p"]).abs()).sum() / s["n"].sum())

def evaluate(y, p, name="model", plot=True, n_bins=10):
    y = np.asarray(y)
    p = np.asarray(p, dtype=float)
    res = {
        "model": name,
        "AUC": roc_auc_score(y, p),
        "logloss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)),
        "ECE": expected_calibration_error(y, p, n_bins),
    }
    if plot:
        frac_pos, mean_pred = calibration_curve(y, p, n_bins=n_bins, strategy="quantile")
        plt.figure(figsize=(4.5, 4.5))
        plt.plot([0, 1], [0, 1], "--", color="gray")
        plt.plot(mean_pred, frac_pos, marker="o")
        plt.xlabel("평균 예측 확률")
        plt.ylabel("실제 재청취 비율")
        plt.title(f"보정 곡선: {name}")
        plt.show()
    return res

def bootstrap_auc_diff(y, user, p_model, p_base, n_boot=300, seed=42):
    y = np.asarray(y).astype(float)
    codes, uniques = pd.factorize(np.asarray(user))
    n_users = len(uniques)
    p_model = np.asarray(p_model, dtype=float)
    p_base = np.asarray(p_base, dtype=float)

    _, g_model = np.unique(p_model, return_inverse=True)
    _, g_base = np.unique(p_base, return_inverse=True)
    k_model = g_model.max() + 1
    k_base = g_base.max() + 1

    def weighted_auc(group, k, w):
        pos = np.bincount(group, weights=w * y, minlength=k)
        neg = np.bincount(group, weights=w * (1 - y), minlength=k)
        below = np.cumsum(neg) - neg
        return float((pos * (below + 0.5 * neg)).sum() / (pos.sum() * neg.sum()))

    ones = np.ones(len(y))
    auc_model = weighted_auc(g_model, k_model, ones)
    auc_base = weighted_auc(g_base, k_base, ones)

    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        draw = rng.integers(0, n_users, n_users)
        cnt = np.bincount(draw, minlength=n_users).astype(float)
        w = cnt[codes]
        diffs[i] = weighted_auc(g_model, k_model, w) - weighted_auc(g_base, k_base, w)

    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {
        "AUC_model": auc_model,
        "AUC_base": auc_base,
        "AUC_diff": auc_model - auc_base,
        "CI_low": float(lo),
        "CI_high": float(hi),
        "pass": bool(auc_model > auc_base and lo > 0),
    }

def check_pass(y, user, p_model, p_base, name="model", n_boot=300):
    r = bootstrap_auc_diff(y, user, p_model, p_base, n_boot=n_boot)
    print(f"[{name}] 모델 AUC {r['AUC_model']:.4f} / 인기도 AUC {r['AUC_base']:.4f}")
    print(f"[{name}] AUC 차이 {r['AUC_diff']:.4f}, 95% CI [{r['CI_low']:.4f}, {r['CI_high']:.4f}]")
    print(f"[{name}] 합격선 충족: {'합격' if r['pass'] else '미충족'}")
    return r

# 사용 예 (valid 평가 시)
# base = popularity_score(valid, df.iloc[:cut(0.6)])
# print(evaluate(valid["target"], base, "인기도", plot=False))
# print(evaluate(valid["target"], p_valid, "LightGBM"))
# check_pass(valid["target"], valid["msno"], p_valid, base, "LightGBM")

합격선 기준 저장 완료


2. 특성 계산 (블록 시작 시점 T 이전 기록으로만)

In [4]:
import numpy as np
import pandas as pd

N_ORIG = 7377418
BLOCKS = {
    "train_a": (0.2, 0.4),
    "train_b": (0.4, 0.6),
    "valid": (0.6, 0.8),
    "test": (0.8, 1.0),
}
LABEL_BOUND = {"train_a": 0.0, "train_b": 0.0, "valid": 0.045, "test": 0.37}
POP_BOUND = {"train_a": 12, "train_b": 21, "valid": 28, "test": 35}
SMOOTH_M = 5

if "df" not in globals():
    df = pd.read_csv("data.csv", engine="pyarrow")
    df = df.sort_values("row_order", kind="stable").reset_index(drop=True)

ro = df["row_order"].to_numpy()

def cut(frac):
    return int(np.searchsorted(ro, frac * N_ORIG, side="left"))

U = pd.factorize(df["msno"])[0].astype(np.int64)
S = pd.factorize(df["song_id"])[0].astype(np.int64)
A = pd.factorize(df["main_artist"])[0].astype(np.int64)
A[df["is_various_artist"].to_numpy() == 1] = -1
pg = df["primary_genre"].astype(str)
G = pd.factorize(pg)[0].astype(np.int64)
G[(pg == "unknown").to_numpy()] = -1
NS, NA, NG = S.max() + 1, A.max() + 1, G.max() + 1
Y = df["target"].to_numpy(dtype=float)

KUS = U * NS + S
KUA = np.where(A >= 0, U * NA + A, -1)
KUG = np.where(G >= 0, U * NG + G, -1)

first = np.unique(U, return_index=True)[1]
mem = df.iloc[first]
days = (pd.to_datetime(mem["expiration_date"]) - pd.to_datetime(mem["registration_init_time"])).dt.days.to_numpy(dtype=float)
MEM_DAYS = np.full(U.max() + 1, np.nan)
MEM_DAYS[U[first]] = days

def cnt_table(keys):
    return pd.Series(keys).value_counts()

def look(tab, keys):
    return np.nan_to_num(tab.reindex(keys).to_numpy(dtype=float))

def ratio(x, total):
    return np.divide(x, total, out=np.zeros_like(x), where=total > 0)

def rate_cols(end, sfx, s0, s1):
    names = ["user_rate", "song_rate", "user_artist_rate", "user_genre_rate"]
    keys = [U, S, KUA, KUG]
    if end == 0:
        return {f"{n}_{sfx}": np.full(s1 - s0, np.nan) for n in names}
    y = Y[:end]
    prior = y.mean()
    out = {}
    for n, key in zip(names, keys):
        kh = key[:end]
        m = kh >= 0
        tab = pd.DataFrame({"k": kh[m], "y": y[m]}).groupby("k")["y"].agg(["sum", "count"])
        s = np.nan_to_num(tab["sum"].reindex(key[s0:s1]).to_numpy(dtype=float))
        c = np.nan_to_num(tab["count"].reindex(key[s0:s1]).to_numpy(dtype=float))
        out[f"{n}_{sfx}"] = (s + prior * SMOOTH_M) / (c + SMOOTH_M)
    return out

def build(name):
    lo, hi = BLOCKS[name]
    s0, s1 = cut(lo), cut(hi)
    e_lab = cut(LABEL_BOUND[name])

    assert s0 > 0 and ro[s0 - 1] < lo * N_ORIG and ro[s0 - 1] < ro[s0]
    assert e_lab <= s0
    if e_lab > 0:
        assert ro[e_lab - 1] < LABEL_BOUND[name] * N_ORIG

    kua_h = KUA[:s0]
    kug_h = KUG[:s0]
    uc = cnt_table(U[:s0])
    sc = cnt_table(S[:s0])
    us = cnt_table(KUS[:s0])
    ua = cnt_table(kua_h[kua_h >= 0])
    ug = cnt_table(kug_h[kug_h >= 0])

    n_song_by_user = us.groupby(us.index.to_numpy() // NS).size()
    n_user_by_song = us.groupby(us.index.to_numpy() % NS).size()
    n_artist_by_user = ua.groupby(ua.index.to_numpy() // NA).size()

    d = pd.DataFrame({"u": ug.index.to_numpy() // NG, "c": ug.to_numpy(dtype=float)})
    d["p"] = d["c"] / d.groupby("u")["c"].transform("sum")
    d["h"] = -d["p"] * np.log(d["p"])
    n_genre_by_user = d.groupby("u").size()
    entropy_by_user = d.groupby("u")["h"].sum()
    topshare_by_user = d.groupby("u")["p"].max()

    sl = slice(s0, s1)
    bu, bs = U[sl], S[sl]
    user_cnt = look(uc, bu)
    song_cnt = look(sc, bs)
    user_song_cnt = look(us, KUS[sl])
    user_artist_cnt = look(ua, KUA[sl])
    user_genre_cnt = look(ug, KUG[sl])

    f = df.iloc[sl][["row_order", "msno", "song_id", "target", "song_length_min", "is_short", "is_long",
                     "n_genres", "n_artists", "language", "is_various_artist", "bd_clean", "age_group"]].reset_index(drop=True)
    f["primary_genre_code"] = G[sl]
    f["gender_code"] = df["gender"].iloc[sl].map({"male": 1, "female": 2}).fillna(0).to_numpy()
    f["membership_days"] = MEM_DAYS[bu]

    f["user_cnt"] = user_cnt
    f["user_nuniq_song"] = look(n_song_by_user, bu)
    f["user_nuniq_artist"] = look(n_artist_by_user, bu)
    f["user_nuniq_genre"] = look(n_genre_by_user, bu)
    f["song_cnt"] = song_cnt
    f["song_nusers"] = look(n_user_by_song, bs)
    f["is_popular"] = (song_cnt >= POP_BOUND[name]).astype(int)
    f["user_song_cnt"] = user_song_cnt
    f["user_artist_cnt"] = user_artist_cnt
    f["user_genre_cnt"] = user_genre_cnt
    f["user_artist_ratio"] = ratio(user_artist_cnt, user_cnt)
    f["user_genre_ratio"] = ratio(user_genre_cnt, user_cnt)
    f["user_genre_entropy"] = look(entropy_by_user, bu)
    f["user_top_genre_share"] = look(topshare_by_user, bu)
    f["is_new_artist"] = ((A[sl] >= 0) & (user_artist_cnt == 0)).astype(int)
    f["is_out_of_taste_genre"] = ((G[sl] >= 0) & (user_genre_cnt == 0)).astype(int)

    for k, v in rate_cols(s0, "g0", s0, s1).items():
        f[k] = v
    for k, v in rate_cols(e_lab, "g30", s0, s1).items():
        f[k] = v

    fc = f.select_dtypes("float64").columns
    f[fc] = f[fc].astype("float32")
    f.to_parquet(f"feat_{name}.parquet", index=False)
    return f

feats = {}
for name in BLOCKS:
    feats[name] = build(name)
    f = feats[name]
    g30_cols = [c for c in f.columns if c.endswith("_g30")]
    print(name, f.shape, "| 이력 끝 위치 < 블록 시작 확인 완료 | g30 결측률:", round(float(f[g30_cols].isna().mean().mean()), 3))

print("특성 열 수:", feats["valid"].shape[1] - 4)

train_a (1470413, 40) | 이력 끝 위치 < 블록 시작 확인 완료 | g30 결측률: 1.0
train_b (1469956, 40) | 이력 끝 위치 < 블록 시작 확인 완료 | g30 결측률: 1.0
valid (1470833, 40) | 이력 끝 위치 < 블록 시작 확인 완료 | g30 결측률: 0.0
test (1471022, 40) | 이력 끝 위치 < 블록 시작 확인 완료 | g30 결측률: 0.0
특성 열 수: 36


3. 기준선

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

tr = pd.read_parquet("feat_train_b.parquet", columns=["target", "song_cnt", "user_rate_g0"])
va = pd.read_parquet("feat_valid.parquet", columns=["msno", "target", "song_cnt", "user_rate_g0", "user_rate_g30"])

def ece(y, p, n_bins=10):
    b = pd.qcut(p, q=n_bins, duplicates="drop")
    s = pd.DataFrame({"y": y, "p": p, "b": b}).groupby("b", observed=True).agg(y=("y", "mean"), p=("p", "mean"), n=("y", "size"))
    return float((s["n"] * (s["y"] - s["p"]).abs()).sum() / s["n"].sum())

def evaluate(y, p, name):
    y = np.asarray(y)
    p = np.asarray(p, dtype=float)
    return {"model": name, "AUC": roc_auc_score(y, p), "logloss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)), "ECE": ece(y, p)}

def bootstrap_auc_diff(y, user, p_model, p_base, n_boot=300, seed=42):
    y = np.asarray(y).astype(float)
    codes, uniques = pd.factorize(np.asarray(user))
    n_users = len(uniques)
    _, g_m = np.unique(np.asarray(p_model, dtype=float), return_inverse=True)
    _, g_b = np.unique(np.asarray(p_base, dtype=float), return_inverse=True)
    k_m, k_b = g_m.max() + 1, g_b.max() + 1

    def wauc(g, k, w):
        pos = np.bincount(g, weights=w * y, minlength=k)
        neg = np.bincount(g, weights=w * (1 - y), minlength=k)
        below = np.cumsum(neg) - neg
        return float((pos * (below + 0.5 * neg)).sum() / (pos.sum() * neg.sum()))

    ones = np.ones(len(y))
    auc_m, auc_b = wauc(g_m, k_m, ones), wauc(g_b, k_b, ones)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        cnt = np.bincount(rng.integers(0, n_users, n_users), minlength=n_users).astype(float)
        w = cnt[codes]
        diffs[i] = wauc(g_m, k_m, w) - wauc(g_b, k_b, w)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"AUC_model": auc_m, "AUC_base": auc_b, "AUC_diff": auc_m - auc_b, "CI_low": float(lo), "CI_high": float(hi), "pass": bool(auc_m > auc_b and lo > 0)}

pop_lr = LogisticRegression()
pop_lr.fit(np.log1p(tr[["song_cnt"]].to_numpy()), tr["target"])
print("인기도 계수(부호 확인):", pop_lr.coef_[0][0])

p_pop = pop_lr.predict_proba(np.log1p(va[["song_cnt"]].to_numpy()))[:, 1]
p_user_g0 = va["user_rate_g0"].to_numpy(dtype=float)
p_user_g30 = va["user_rate_g30"].to_numpy(dtype=float)

res = pd.DataFrame([
    evaluate(va["target"], p_pop, "인기도(곡 청취 수)"),
    evaluate(va["target"], p_user_g0, "사용자 평균 재청취율(g0)"),
    evaluate(va["target"], p_user_g30, "사용자 평균 재청취율(g30)"),
])
print(res.round(4).to_string(index=False))

r = bootstrap_auc_diff(va["target"], va["msno"], p_user_g0, p_pop)
print(f"사용자 기준선(g0) vs 인기도: AUC 차이 {r['AUC_diff']:.4f}, 95% CI [{r['CI_low']:.4f}, {r['CI_high']:.4f}]")

pd.DataFrame({"p_pop": p_pop, "p_user_g0": p_user_g0}).to_parquet("baseline_valid_pred.parquet", index=False)

/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights


인기도 계수(부호 확인): 0.11431298920514935
           model    AUC  logloss    ECE
     인기도(곡 청취 수) 0.5631   0.6834 0.0601
 사용자 평균 재청취율(g0) 0.6121   0.7226 0.1312
사용자 평균 재청취율(g30) 0.5438   0.9727 0.2963
사용자 기준선(g0) vs 인기도: AUC 차이 0.0490, 95% CI [0.0438, 0.0541]


4. 파생변수


4-1. 구간별 재청취율, 탐색 성향 특성 추가

In [12]:
import numpy as np
import pandas as pd

N_ORIG = 7377418
BLOCKS = {
    "train_a": (0.2, 0.4),
    "train_b": (0.4, 0.6),
    "valid": (0.6, 0.8),
    "test": (0.8, 1.0),
}
LABEL_BOUND = {"train_a": 0.0, "train_b": 0.0, "valid": 0.045, "test": 0.37}
POP_BOUND = {"train_a": 12, "train_b": 21, "valid": 28, "test": 35}
SMOOTH_M = 5
SAVE_CSV = False

if "df" not in globals():
    df = pd.read_csv("data.csv", engine="pyarrow")
    df = df.sort_values("row_order", kind="stable").reset_index(drop=True)

ro = df["row_order"].to_numpy()

def cut(frac):
    return int(np.searchsorted(ro, frac * N_ORIG, side="left"))

U = pd.factorize(df["msno"])[0].astype(np.int64)
S = pd.factorize(df["song_id"])[0].astype(np.int64)
A = pd.factorize(df["main_artist"])[0].astype(np.int64)
A[df["is_various_artist"].to_numpy() == 1] = -1
pg = df["primary_genre"].astype(str)
G = pd.factorize(pg)[0].astype(np.int64)
G[(pg == "unknown").to_numpy()] = -1
Y = df["target"].to_numpy(dtype=float)

nU = int(U.max()) + 1
NS = int(S.max()) + 1
NA = int(A.max()) + 1
NG = int(G.max()) + 1

KUS = U * NS + S
KUA = np.where(A >= 0, U * NA + A, -1)
KUG = np.where(G >= 0, U * NG + G, -1)

def prior_cnt(keys):
    return pd.Series(keys).groupby(keys).cumcount().to_numpy()

prior_song = prior_cnt(S)
prior_us = prior_cnt(KUS)
prior_ua = prior_cnt(KUA)
prior_ug = prior_cnt(KUG)

valid_a = A >= 0
valid_g = G >= 0
new_a = valid_a & (prior_ua == 0)
oot_g = valid_g & (prior_ug == 0)
rep_song = prior_us > 0
log_pop = np.log1p(prior_song)

def user_count(end, mask):
    return np.bincount(U[:end][mask[:end]], minlength=nU).astype(float)

def safe_ratio(a, b):
    out = np.full(len(a), np.nan)
    np.divide(a, b, out=out, where=b > 0)
    return out

def seg_rate(end, seg, s0, s1):
    if end == 0:
        return np.full(s1 - s0, np.nan)
    m = seg[:end]
    u = U[:end][m]
    y = Y[:end][m]
    prior = y.mean() if len(y) > 0 else Y[:end].mean()
    ssum = np.bincount(u, weights=y, minlength=nU)
    scnt = np.bincount(u, minlength=nU).astype(float)
    bu = U[s0:s1]
    return (ssum[bu] + prior * SMOOTH_M) / (scnt[bu] + SMOOTH_M)

def build(name):
    lo, hi = BLOCKS[name]
    s0, s1 = cut(lo), cut(hi)
    e_lab = cut(LABEL_BOUND[name])

    assert s0 > 0 and ro[s0 - 1] < lo * N_ORIG and ro[s0 - 1] < ro[s0]
    assert e_lab <= s0
    if e_lab > 0:
        assert ro[e_lab - 1] < LABEL_BOUND[name] * N_ORIG

    bu = U[s0:s1]
    pop_seg = prior_song >= POP_BOUND[name]
    unpop_seg = ~pop_seg

    song_cnt_T = np.bincount(S[:s0], minlength=NS)[S[s0:s1]]
    is_pop_row = song_cnt_T >= POP_BOUND[name]

    f = pd.DataFrame({
        "row_order": ro[s0:s1],
        "msno": df["msno"].to_numpy()[s0:s1],
        "song_id": df["song_id"].to_numpy()[s0:s1],
    })

    segs = {
        "user_pop_rate": pop_seg,
        "user_unpop_rate": unpop_seg,
        "user_newartist_rate": new_a,
        "user_ootgenre_rate": oot_g,
    }
    for sfx, end in (("g0", s0), ("g30", e_lab)):
        rates = {}
        for col, seg in segs.items():
            rates[col] = seg_rate(end, seg, s0, s1)
            f[f"{col}_{sfx}"] = rates[col]
        f[f"user_popseg_rate_{sfx}"] = np.where(is_pop_row, rates["user_pop_rate"], rates["user_unpop_rate"])

    n_hist = np.bincount(U[:s0], minlength=nU).astype(float)
    f["user_pop_n"] = user_count(s0, pop_seg)[bu]
    f["user_unpop_n"] = user_count(s0, unpop_seg)[bu]
    f["user_newartist_n"] = user_count(s0, new_a)[bu]
    f["user_ootgenre_n"] = user_count(s0, oot_g)[bu]

    f["user_new_artist_ratio"] = safe_ratio(user_count(s0, new_a), user_count(s0, valid_a))[bu]
    f["user_new_genre_ratio"] = safe_ratio(user_count(s0, oot_g), user_count(s0, valid_g))[bu]
    f["user_unpop_ratio"] = safe_ratio(user_count(s0, unpop_seg), n_hist)[bu]
    f["user_repeat_song_ratio"] = safe_ratio(user_count(s0, rep_song), n_hist)[bu]
    pop_sum = np.bincount(U[:s0], weights=log_pop[:s0], minlength=nU)
    f["user_avg_song_pop"] = safe_ratio(pop_sum, n_hist)[bu]

    fc = f.select_dtypes("float64").columns
    f[fc] = f[fc].astype("float32")

    old = pd.read_parquet(f"feat_{name}.parquet")
    assert len(old) == len(f) and (old["row_order"].to_numpy() == f["row_order"].to_numpy()).all()
    new_cols = [c for c in f.columns if c not in ("row_order", "msno", "song_id")]
    merged = pd.concat([old.reset_index(drop=True), f[new_cols]], axis=1)

    f.to_parquet(f"feat2_new_{name}.parquet", index=False)
    merged.to_parquet(f"featv2_{name}.parquet", index=False)
    if SAVE_CSV:
        f.to_csv(f"feat2_new_{name}.csv", index=False)
        merged.to_csv(f"featv2_{name}.csv", index=False)
    return f, merged

new_feats = {}
for name in BLOCKS:
    f, merged = build(name)
    new_feats[name] = f
    new_cols = [c for c in f.columns if c not in ("row_order", "msno", "song_id")]
    g30_cols = [c for c in new_cols if c.endswith("_g30")]
    nan_rate = f[new_cols].isna().mean().round(3)
    print(name, "| 새 특성", len(new_cols), "| 합친 파일", merged.shape, "| g30 결측률", round(float(f[g30_cols].isna().mean().mean()), 3))

print(new_feats["valid"][new_cols].describe().T[["mean", "min", "max"]].round(4).to_string())

train_a | 새 특성 19 | 합친 파일 (1470413, 59) | g30 결측률 1.0
train_b | 새 특성 19 | 합친 파일 (1469956, 59) | g30 결측률 1.0
valid | 새 특성 19 | 합친 파일 (1470833, 59) | g30 결측률 0.0
test | 새 특성 19 | 합친 파일 (1471022, 59) | g30 결측률 0.0
                             mean     min        max
user_pop_rate_g0           0.5777  0.0372     0.9947
user_unpop_rate_g0         0.4878  0.0136     0.9912
user_newartist_rate_g0     0.5424  0.0375     0.9907
user_ootgenre_rate_g0      0.5561  0.0970     0.9122
user_popseg_rate_g0        0.5566  0.0136     0.9947
user_pop_rate_g30          0.7989  0.0508     0.9923
user_unpop_rate_g30        0.6853  0.0568     0.9927
user_newartist_rate_g30    0.7258  0.0567     0.9888
user_ootgenre_rate_g30     0.7267  0.2025     0.9361
user_popseg_rate_g30       0.7668  0.0508     0.9927
user_pop_n               227.5222  0.0000  2054.0000
user_unpop_n             123.3183  0.0000  2449.0000
user_newartist_n         147.3023  0.0000  1602.0000
user_ootgenre_n           12.6234  0.0000    44

In [14]:
## 비교 
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, log_loss

SEED = 42
SAMPLE_N = 500000
CAT = "primary_genre_code"
KEYS = {"row_order", "msno", "song_id", "target"}
DROP_NEW = {"user_repeat_song_ratio"}

def load(prefix):
    tr = pd.concat([pd.read_parquet(f"{prefix}_train_a.parquet"), pd.read_parquet(f"{prefix}_train_b.parquet")], ignore_index=True)
    tr = tr.sample(n=SAMPLE_N, random_state=SEED).reset_index(drop=True)
    va = pd.read_parquet(f"{prefix}_valid.parquet")
    return tr, va

tr_old, va_old = load("feat")
tr_new, va_new = load("featv2")

y = va_old["target"].to_numpy()
users = va_old["msno"].unique()
rng = np.random.default_rng(SEED)
tune_users = set(rng.choice(users, len(users) // 2, replace=False))
is_tune = va_old["msno"].isin(tune_users).to_numpy()
is_hold = ~is_tune
yt, yh = y[is_tune], y[is_hold]

def feats(va, extra_drop=()):
    drop = KEYS | {c for c in va.columns if c.endswith("_g30")} | set(extra_drop)
    return [c for c in va.columns if c not in drop]

def prep(tr, va, fs):
    A = tr[fs].copy()
    B = va[fs].copy()
    A[CAT] = A[CAT].astype(int) + 1
    B[CAT] = B[CAT].astype(int) + 1
    return A, B, tr["target"].to_numpy()

def run(Xtr, ytr, Xva):
    Xt, Xh = Xva[is_tune], Xva[is_hold]
    out = {}

    t0 = time.time()
    m = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.06, num_leaves=63, max_depth=8, min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=2.0, n_jobs=-1, random_state=SEED, verbose=-1)
    m.fit(Xtr, ytr, eval_set=[(Xt, yt)], eval_metric="auc", categorical_feature=[CAT], callbacks=[lgb.early_stopping(30, verbose=False)])
    out["LightGBM"] = m.predict_proba(Xva, num_iteration=m.best_iteration_)[:, 1]
    imp = pd.Series(m.booster_.feature_importance(importance_type="gain"), index=Xtr.columns)
    imp = (imp / imp.sum()).sort_values(ascending=False)

    m = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.06, max_depth=7, min_child_weight=50, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, tree_method="hist", n_jobs=-1, random_state=SEED, eval_metric="auc", early_stopping_rounds=30)
    m.fit(Xtr, ytr, eval_set=[(Xt, yt)], verbose=False)
    out["XGBoost"] = m.predict_proba(Xva, iteration_range=(0, m.best_iteration + 1))[:, 1]

    m = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=7, l2_leaf_reg=3, random_seed=SEED, verbose=0, thread_count=-1, cat_features=[CAT], eval_metric="AUC", od_type="Iter", od_wait=30, use_best_model=True)
    m.fit(Xtr, ytr, eval_set=(Xt, yt))
    out["CatBoost"] = m.predict_proba(Xva)[:, 1]

    m = HistGradientBoostingClassifier(learning_rate=0.08, max_iter=250, max_leaf_nodes=63, min_samples_leaf=200, l2_regularization=1.0, random_state=SEED)
    m.fit(Xtr, ytr)
    out["HistGradientBoosting"] = m.predict_proba(Xva)[:, 1]
    print(f"학습 {time.time() - t0:.0f}초")
    return out, imp

f_old = feats(va_old)
f_new = feats(va_new, DROP_NEW)
new_only = [c for c in f_new if c not in f_old]
print("기존 특성", len(f_old), "| 새 버전 특성", len(f_new), "| 추가", len(new_only))

Xa, Xb, ya = prep(tr_old, va_old, f_old)
p_old, imp_old = run(Xa, ya, Xb)
Xa, Xb, ya = prep(tr_new, va_new, f_new)
p_new, imp_new = run(Xa, ya, Xb)

rows = []
for n in p_old:
    ho, hn = roc_auc_score(yh, p_old[n][is_hold]), roc_auc_score(yh, p_new[n][is_hold])
    to, tn = roc_auc_score(yt, p_old[n][is_tune]), roc_auc_score(yt, p_new[n][is_tune])
    rows.append({
        "model": n,
        "holdout_기존": ho, "holdout_새": hn, "holdout_차이": hn - ho,
        "tune_기존": to, "tune_새": tn,
        "logloss_기존": log_loss(yh, np.clip(p_old[n][is_hold], 1e-6, 1 - 1e-6)),
        "logloss_새": log_loss(yh, np.clip(p_new[n][is_hold], 1e-6, 1 - 1e-6)),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))

print("새 특성 중요도 (LightGBM gain, 새 버전)")
print(imp_new[imp_new.index.isin(new_only)].round(4).to_string())
print("새 특성 중요도 합:", round(float(imp_new[imp_new.index.isin(new_only)].sum()), 4))

pd.DataFrame(p_new).to_parquet("pred_valid_models_v2.parquet", index=False)

기존 특성 32 | 새 버전 특성 45 | 추가 13
학습 32초
학습 36초
               model  holdout_기존  holdout_새  holdout_차이  tune_기존  tune_새  logloss_기존  logloss_새
            LightGBM      0.6610     0.6636      0.0026   0.6621  0.6643      0.6421     0.6409
             XGBoost      0.6610     0.6627      0.0018   0.6613  0.6633      0.6420     0.6414
            CatBoost      0.6605     0.6621      0.0015   0.6610  0.6627      0.6424     0.6416
HistGradientBoosting      0.6593     0.6618      0.0024   0.6590  0.6611      0.6435     0.6423
새 특성 중요도 (LightGBM gain, 새 버전)
user_popseg_rate_g0       0.2230
user_newartist_rate_g0    0.0254
user_avg_song_pop         0.0198
user_pop_rate_g0          0.0147
user_unpop_rate_g0        0.0118
user_new_artist_ratio     0.0110
user_new_genre_ratio      0.0091
user_pop_n                0.0081
user_unpop_ratio          0.0077
user_ootgenre_rate_g0     0.0068
user_unpop_n              0.0056
user_newartist_n          0.0010
user_ootgenre_n           0.0008
새 특성 중요도 합: 0.34

5. 모델 선택 (앙상블)

In [16]:
import itertools
import os
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, log_loss

SEED = 42
SAMPLE_N = 500000
SAMPLE_SLOW = 200000
MIN_K, MAX_K = 2, 5
PREFIX = "featv2"
PRED_FILE = "pred_valid_models_v2_13.parquet"
DROP_NEW = {"user_repeat_song_ratio"}

va = pd.read_parquet(f"{PREFIX}_valid.parquet")
base = pd.read_parquet("baseline_valid_pred.parquet")

drop_cols = {"row_order", "msno", "song_id", "target"} | {c for c in va.columns if c.endswith("_g30")} | DROP_NEW
FEATS = [c for c in va.columns if c not in drop_cols]
CAT = "primary_genre_code"
print("사용 특성 수:", len(FEATS))

y = va["target"].to_numpy()
users = va["msno"].unique()
rng = np.random.default_rng(SEED)
tune_users = set(rng.choice(users, len(users) // 2, replace=False))
is_tune = va["msno"].isin(tune_users).to_numpy()
is_hold = ~is_tune
yt, yh = y[is_tune], y[is_hold]
user_h = va["msno"].to_numpy()[is_hold]
pop_h = base["p_pop"].to_numpy()[is_hold]
usr_h = base["p_user_g0"].to_numpy()[is_hold]

def ece(yy, p, n_bins=10):
    b = pd.qcut(p, q=n_bins, duplicates="drop")
    s = pd.DataFrame({"y": yy, "p": p, "b": b}).groupby("b", observed=True).agg(y=("y", "mean"), p=("p", "mean"), n=("y", "size"))
    return float((s["n"] * (s["y"] - s["p"]).abs()).sum() / s["n"].sum())

def bootstrap_auc_diff(yy, user, p_model, p_base, n_boot=100, seed=42):
    yy = np.asarray(yy).astype(float)
    codes, uniques = pd.factorize(np.asarray(user))
    n_users = len(uniques)
    _, g_m = np.unique(np.asarray(p_model, dtype=float), return_inverse=True)
    _, g_b = np.unique(np.asarray(p_base, dtype=float), return_inverse=True)
    k_m, k_b = g_m.max() + 1, g_b.max() + 1

    def wauc(g, k, w):
        pos = np.bincount(g, weights=w * yy, minlength=k)
        neg = np.bincount(g, weights=w * (1 - yy), minlength=k)
        below = np.cumsum(neg) - neg
        return float((pos * (below + 0.5 * neg)).sum() / (pos.sum() * neg.sum()))

    ones = np.ones(len(yy))
    auc_m, auc_b = wauc(g_m, k_m, ones), wauc(g_b, k_b, ones)
    r = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        cnt = np.bincount(r.integers(0, n_users, n_users), minlength=n_users).astype(float)
        w = cnt[codes]
        diffs[i] = wauc(g_m, k_m, w) - wauc(g_b, k_b, w)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return auc_m - auc_b, float(lo), float(hi)

if os.path.exists(PRED_FILE):
    P = pd.read_parquet(PRED_FILE)
    print("저장된 예측 파일을 불러왔습니다:", list(P.columns))
else:
    tr_all = pd.concat([pd.read_parquet(f"{PREFIX}_train_a.parquet"), pd.read_parquet(f"{PREFIX}_train_b.parquet")], ignore_index=True)
    tr = tr_all.sample(n=SAMPLE_N, random_state=SEED).reset_index(drop=True)
    del tr_all
    ytr = tr["target"].to_numpy()

    Xtr = tr[FEATS].copy()
    Xva = va[FEATS].copy()
    Xtr[CAT] = Xtr[CAT].astype(int) + 1
    Xva[CAT] = Xva[CAT].astype(int) + 1
    Xt, Xh = Xva[is_tune], Xva[is_hold]

    sub = np.random.default_rng(SEED).choice(len(tr), SAMPLE_SLOW, replace=False)
    Xtr_fill, Xva_fill = Xtr.fillna(-1), Xva.fillna(-1)

    count_cols = [c for c in FEATS if c.endswith("_cnt") or c.endswith("_n") or c.startswith("user_nuniq") or c in ("song_nusers", "membership_days")]
    lr_cols = [c for c in FEATS if c != CAT]

    A = tr[lr_cols].astype("float64").replace([np.inf, -np.inf], np.nan)
    B = va[lr_cols].astype("float64").replace([np.inf, -np.inf], np.nan)
    for c in count_cols:
        A[c] = np.log1p(A[c].clip(lower=0))
        B[c] = np.log1p(B[c].clip(lower=0))
    med = A.median()
    A, B = A.fillna(med), B.fillna(med)
    lo_q, hi_q = A.quantile(0.001), A.quantile(0.999)
    A, B = A.clip(lo_q, hi_q, axis=1), B.clip(lo_q, hi_q, axis=1)
    sc = StandardScaler().fit(A)
    Za = np.clip(sc.transform(A), -8, 8)
    Zb = np.clip(sc.transform(B), -8, 8)

    def fit_lr():
        return LogisticRegression(C=1.0, max_iter=300).fit(Za, ytr).predict_proba(Zb)[:, 1]

    def fit_sgd():
        m = SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=30, tol=1e-4, random_state=SEED)
        return m.fit(Za, ytr).predict_proba(Zb)[:, 1]

    def fit_nb():
        return GaussianNB().fit(Za, ytr).predict_proba(Zb)[:, 1]

    def fit_rf():
        m = RandomForestClassifier(n_estimators=80, max_depth=16, min_samples_leaf=100, max_features=0.5, n_jobs=-1, random_state=SEED)
        m.fit(Xtr_fill.iloc[sub], ytr[sub])
        return m.predict_proba(Xva_fill)[:, 1]

    def fit_et():
        m = ExtraTreesClassifier(n_estimators=100, max_depth=18, min_samples_leaf=100, max_features=0.5, n_jobs=-1, random_state=SEED)
        m.fit(Xtr_fill.iloc[sub], ytr[sub])
        return m.predict_proba(Xva_fill)[:, 1]

    def fit_ada():
        base_tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200)
        m = AdaBoostClassifier(estimator=base_tree, n_estimators=60, learning_rate=0.5, random_state=SEED)
        m.fit(Xtr_fill.iloc[sub], ytr[sub])
        return m.predict_proba(Xva_fill)[:, 1]

    def fit_hgb():
        m = HistGradientBoostingClassifier(learning_rate=0.08, max_iter=250, max_leaf_nodes=63, min_samples_leaf=200, l2_regularization=1.0, random_state=SEED)
        return m.fit(Xtr, ytr).predict_proba(Xva)[:, 1]

    def fit_lgbm():
        m = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.06, num_leaves=63, max_depth=8, min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=2.0, n_jobs=-1, random_state=SEED, verbose=-1)
        m.fit(Xtr, ytr, eval_set=[(Xt, yt)], eval_metric="auc", categorical_feature=[CAT], callbacks=[lgb.early_stopping(30, verbose=False)])
        return m.predict_proba(Xva, num_iteration=m.best_iteration_)[:, 1]

    def fit_lgbm2():
        m = lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.03, num_leaves=31, max_depth=6, min_child_samples=500, subsample=0.7, subsample_freq=1, colsample_bytree=0.6, reg_lambda=10.0, n_jobs=-1, random_state=SEED + 1, verbose=-1)
        m.fit(Xtr, ytr, eval_set=[(Xt, yt)], eval_metric="auc", categorical_feature=[CAT], callbacks=[lgb.early_stopping(40, verbose=False)])
        return m.predict_proba(Xva, num_iteration=m.best_iteration_)[:, 1]

    def fit_xgb():
        m = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.06, max_depth=7, min_child_weight=50, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, tree_method="hist", n_jobs=-1, random_state=SEED, eval_metric="auc", early_stopping_rounds=30)
        m.fit(Xtr, ytr, eval_set=[(Xt, yt)], verbose=False)
        return m.predict_proba(Xva, iteration_range=(0, m.best_iteration + 1))[:, 1]

    def fit_xgb2():
        m = xgb.XGBClassifier(n_estimators=1500, learning_rate=0.03, max_depth=4, min_child_weight=200, subsample=0.7, colsample_bytree=0.6, reg_lambda=10.0, tree_method="hist", n_jobs=-1, random_state=SEED + 1, eval_metric="auc", early_stopping_rounds=40)
        m.fit(Xtr, ytr, eval_set=[(Xt, yt)], verbose=False)
        return m.predict_proba(Xva, iteration_range=(0, m.best_iteration + 1))[:, 1]

    def fit_cat():
        m = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=7, l2_leaf_reg=3, random_seed=SEED, verbose=0, thread_count=-1, cat_features=[CAT], eval_metric="AUC", od_type="Iter", od_wait=30, use_best_model=True)
        m.fit(Xtr, ytr, eval_set=(Xt, yt))
        return m.predict_proba(Xva)[:, 1]

    def fit_cat2():
        m = CatBoostClassifier(iterations=1500, learning_rate=0.06, depth=5, l2_leaf_reg=10, random_seed=SEED + 1, verbose=0, thread_count=-1, cat_features=[CAT], eval_metric="AUC", od_type="Iter", od_wait=40, use_best_model=True)
        m.fit(Xtr, ytr, eval_set=(Xt, yt))
        return m.predict_proba(Xva)[:, 1]

    CANDIDATES = {
        "LogisticRegression": fit_lr,
        "SGD_Logistic": fit_sgd,
        "GaussianNB": fit_nb,
        "RandomForest": fit_rf,
        "ExtraTrees": fit_et,
        "AdaBoost": fit_ada,
        "HistGradientBoosting": fit_hgb,
        "LightGBM": fit_lgbm,
        "LightGBM_v2": fit_lgbm2,
        "XGBoost": fit_xgb,
        "XGBoost_v2": fit_xgb2,
        "CatBoost": fit_cat,
        "CatBoost_v2": fit_cat2,
    }

    preds = {}
    for name, fn in CANDIDATES.items():
        t0 = time.time()
        try:
            preds[name] = fn()
        except Exception as e:
            print(f"[{name}] 건너뜀: {type(e).__name__}: {e}")
            continue
        print(f"[{name}] 완료 {time.time() - t0:.0f}초 | holdout AUC {roc_auc_score(yh, preds[name][is_hold]):.4f}")

    P = pd.DataFrame(preds)
    P.to_parquet(PRED_FILE, index=False)
    print("예측 저장 완료:", PRED_FILE)

names = list(P.columns)
print("사용 모델 수:", len(names), names)

PA = P.to_numpy(dtype=float)
PT, PH = PA[is_tune], PA[is_hold]

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

ZA = logit(PA)
ZT, ZH = ZA[is_tune], ZA[is_hold]

def stack_fit(idx):
    return LogisticRegression(C=1.0, max_iter=200).fit(ZT[:, idx], yt)

t0 = time.time()
rows = []
for k in range(MIN_K, min(MAX_K, len(names)) + 1):
    for combo in itertools.combinations(range(len(names)), k):
        idx = list(combo)
        label = " + ".join(names[i] for i in idx)
        rows.append({"조합": label, "개수": k, "방식": "평균", "idx": idx, "tune_AUC": roc_auc_score(yt, PT[:, idx].mean(axis=1))})
        m = stack_fit(idx)
        rows.append({"조합": label, "개수": k, "방식": "스태킹", "idx": idx, "tune_AUC": roc_auc_score(yt, m.predict_proba(ZT[:, idx])[:, 1])})

res = pd.DataFrame(rows).sort_values("tune_AUC", ascending=False).reset_index(drop=True)
print(f"조합 {len(res) // 2}개 x 방식 2개 평가 완료 ({time.time() - t0:.0f}초)")

singles = pd.DataFrame([{"조합": n, "개수": 1, "방식": "단일", "idx": [i], "tune_AUC": roc_auc_score(yt, PT[:, i])} for i, n in enumerate(names)]).sort_values("tune_AUC", ascending=False)

def predict_hold(row):
    idx = row["idx"]
    if row["방식"] == "평균":
        return PH[:, idx].mean(axis=1)
    if row["방식"] == "스태킹":
        return stack_fit(idx).predict_proba(ZH[:, idx])[:, 1]
    return PH[:, idx[0]]

single_rows = []
for _, row in singles.iterrows():
    ph = predict_hold(row)
    single_rows.append({"모델": row["조합"], "tune_AUC": row["tune_AUC"], "holdout_AUC": roc_auc_score(yh, ph),
                        "logloss": log_loss(yh, np.clip(ph, 1e-6, 1 - 1e-6)), "ECE": ece(yh, ph)})
print("단일 모델 성능")
print(pd.DataFrame(single_rows).round(4).to_string(index=False))

top = pd.concat([res.head(10), singles.head(1)], ignore_index=True)
out = []
for _, row in top.iterrows():
    ph = predict_hold(row)
    out.append({
        "조합": row["조합"], "방식": row["방식"], "tune_AUC": row["tune_AUC"],
        "holdout_AUC": roc_auc_score(yh, ph),
        "logloss": log_loss(yh, np.clip(ph, 1e-6, 1 - 1e-6)),
        "ECE": ece(yh, ph),
    })
print("상위 10개 조합과 최고 단일 모델의 holdout 성능")
print(pd.DataFrame(out).round(4).to_string(index=False))

best = res.iloc[0]
best_h = predict_hold(best)
best_single = singles.iloc[0]
single_h = predict_hold(best_single)

print("\n선정된 앙상블:", best["조합"], "|", best["방식"], "| 모델 수", best["개수"])
d, lo, hi = bootstrap_auc_diff(yh, user_h, best_h, pop_h)
print(f"인기도 대비 AUC 차이 {d:.4f}, 95% CI [{lo:.4f}, {hi:.4f}] -> {'합격' if d > 0 and lo > 0 else '미충족'}")
d, lo, hi = bootstrap_auc_diff(yh, user_h, best_h, usr_h)
print(f"사용자 기준선 대비 AUC 차이 {d:.4f}, 95% CI [{lo:.4f}, {hi:.4f}]")
d, lo, hi = bootstrap_auc_diff(yh, user_h, best_h, single_h)
print(f"최고 단일 모델({best_single['조합']}) 대비 AUC 차이 {d:.5f}, 95% CI [{lo:.5f}, {hi:.5f}]")

idx = list(best["idx"])
full = PA[:, idx].mean(axis=1) if best["방식"] == "평균" else stack_fit(idx).predict_proba(ZA[:, idx])[:, 1]
pd.DataFrame({"p_ensemble": full}).to_parquet("pred_valid_ensemble_best_v2_13.parquet", index=False)
res.drop(columns=["idx"]).to_csv("ensemble_combo_results_v2_13.csv", index=False)
print("저장 완료: pred_valid_ensemble_best_v2_13.parquet / ensemble_combo_results_v2_13.csv")

사용 특성 수: 45


/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_

[LogisticRegression] 완료 1초 | holdout AUC 0.6517


/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


[SGD_Logistic] 완료 3초 | holdout AUC 0.6520
[GaussianNB] 완료 0초 | holdout AUC 0.6196
[RandomForest] 완료 8초 | holdout AUC 0.6561
[ExtraTrees] 완료 3초 | holdout AUC 0.6537
[AdaBoost] 완료 55초 | holdout AUC 0.6518
[HistGradientBoosting] 완료 8초 | holdout AUC 0.6618
[LightGBM] 완료 6초 | holdout AUC 0.6636
[LightGBM_v2] 완료 14초 | holdout AUC 0.6644
[XGBoost] 완료 13초 | holdout AUC 0.6627
[XGBoost_v2] 완료 60초 | holdout AUC 0.6630
[CatBoost] 완료 9초 | holdout AUC 0.6621
[CatBoost_v2] 완료 25초 | holdout AUC 0.6635
예측 저장 완료: pred_valid_models_v2_13.parquet
사용 모델 수: 13 ['LogisticRegression', 'SGD_Logistic', 'GaussianNB', 'RandomForest', 'ExtraTrees', 'AdaBoost', 'HistGradientBoosting', 'LightGBM', 'LightGBM_v2', 'XGBoost', 'XGBoost_v2', 'CatBoost', 'CatBoost_v2']


/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid val

... (4 outputs left out)


선정된 앙상블: GaussianNB + AdaBoost + LightGBM + LightGBM_v2 + CatBoost_v2 | 스태킹 | 모델 수 5
인기도 대비 AUC 차이 0.1030, 95% CI [0.0980, 0.1074] -> 합격
사용자 기준선 대비 AUC 차이 0.0546, 95% CI [0.0519, 0.0581]
최고 단일 모델(LightGBM_v2) 대비 AUC 차이 0.00114, 95% CI [0.00059, 0.00160]


/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept


저장 완료: pred_valid_ensemble_best_v2_13.parquet / ensemble_combo_results_v2_13.csv


/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


5. 구간별 성능 확인

In [18]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

SEED = 42
N_BOOT = 200
MEMBERS = ["GaussianNB", "AdaBoost", "LightGBM_v2", "CatBoost_v2"]

va = pd.read_parquet("featv2_valid.parquet", columns=["msno", "target", "is_popular", "is_new_artist", "is_out_of_taste_genre", "is_various_artist", "user_cnt"])
base = pd.read_parquet("baseline_valid_pred.parquet")
P = pd.read_parquet("pred_valid_models_v2_13.parquet")

y = va["target"].to_numpy()
users = va["msno"].unique()
rng = np.random.default_rng(SEED)
tune_users = set(rng.choice(users, len(users) // 2, replace=False))
is_tune = va["msno"].isin(tune_users).to_numpy()
is_hold = ~is_tune

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

Z = np.column_stack([logit(P[m]) for m in MEMBERS])
meta = LogisticRegression(C=1.0, max_iter=200).fit(Z[is_tune], y[is_tune])
pA_all = meta.predict_proba(Z)[:, 1]
print("스태킹 계수:", dict(zip(MEMBERS, meta.coef_[0].round(3))), "절편:", round(float(meta.intercept_[0]), 3))
pd.DataFrame({"p_modelA": pA_all}).to_parquet("pred_valid_modelA.parquet", index=False)

h = va[is_hold].reset_index(drop=True)
yh = y[is_hold]
preds = {
    "모델A": pA_all[is_hold],
    "LightGBM_v2": P["LightGBM_v2"].to_numpy()[is_hold],
    "인기도": base["p_pop"].to_numpy()[is_hold],
    "사용자기준선": base["p_user_g0"].to_numpy()[is_hold],
}

segments = {
    "전체": np.ones(len(h), dtype=bool),
    "인기곡": h["is_popular"].to_numpy() == 1,
    "비인기곡": h["is_popular"].to_numpy() == 0,
    "새 아티스트": h["is_new_artist"].to_numpy() == 1,
    "기존 아티스트": h["is_new_artist"].to_numpy() == 0,
    "취향 밖 장르": h["is_out_of_taste_genre"].to_numpy() == 1,
    "취향 안 장르": h["is_out_of_taste_genre"].to_numpy() == 0,
    "비인기곡+새 아티스트": (h["is_popular"].to_numpy() == 0) & (h["is_new_artist"].to_numpy() == 1),
    "Various Artists": h["is_various_artist"].to_numpy() == 1,
}

def seg_eval(mask, name):
    ys = yh[mask].astype(float)
    us = h["msno"].to_numpy()[mask]
    codes, uniq = pd.factorize(us)
    nu = len(uniq)
    grp, kk = {}, {}
    for k, p in preds.items():
        _, g = np.unique(p[mask], return_inverse=True)
        grp[k], kk[k] = g, g.max() + 1

    def wauc(k, w):
        pos = np.bincount(grp[k], weights=w * ys, minlength=kk[k])
        neg = np.bincount(grp[k], weights=w * (1 - ys), minlength=kk[k])
        below = np.cumsum(neg) - neg
        return float((pos * (below + 0.5 * neg)).sum() / (pos.sum() * neg.sum()))

    ones = np.ones(len(ys))
    auc = {k: wauc(k, ones) for k in preds}
    r = np.random.default_rng(SEED)
    boot = {k: np.empty(N_BOOT) for k in preds}
    for i in range(N_BOOT):
        cnt = np.bincount(r.integers(0, nu, nu), minlength=nu).astype(float)
        w = cnt[codes]
        for k in preds:
            boot[k][i] = wauc(k, w)

    def ci(a):
        return np.percentile(a, [2.5, 97.5])

    row = {"구간": name, "행수": int(mask.sum()), "사용자수": nu, "target평균": round(float(ys.mean()), 4)}
    for k in preds:
        row[f"AUC_{k}"] = auc[k]
    lo, hi = ci(boot["모델A"])
    row["모델A_CI하한"], row["모델A_CI상한"] = lo, hi
    for k in ["인기도", "LightGBM_v2", "사용자기준선"]:
        d = boot["모델A"] - boot[k]
        l, u = ci(d)
        row[f"차이_vs_{k}"] = auc["모델A"] - auc[k]
        row[f"차이하한_vs_{k}"] = l
        row[f"차이상한_vs_{k}"] = u
    row["인기도대비_합격"] = bool(auc["모델A"] > auc["인기도"] and row["차이하한_vs_인기도"] > 0)
    return row

rows = [seg_eval(m, n) for n, m in segments.items() if m.sum() > 1000 and 0 < yh[m].mean() < 1]
res = pd.DataFrame(rows)

print("구간별 규모와 AUC")
print(res[["구간", "행수", "사용자수", "target평균", "AUC_모델A", "모델A_CI하한", "모델A_CI상한", "AUC_LightGBM_v2", "AUC_인기도", "AUC_사용자기준선"]].round(4).to_string(index=False))
print("\n모델A 차이와 95% 신뢰구간")
print(res[["구간", "차이_vs_인기도", "차이하한_vs_인기도", "차이상한_vs_인기도", "인기도대비_합격", "차이_vs_LightGBM_v2", "차이하한_vs_LightGBM_v2", "차이상한_vs_LightGBM_v2", "차이_vs_사용자기준선"]].round(4).to_string(index=False))

ll = []
for n, m in segments.items():
    if m.sum() > 1000 and 0 < yh[m].mean() < 1:
        p = np.clip(preds["모델A"][m], 1e-6, 1 - 1e-6)
        ll.append({"구간": n, "logloss": log_loss(yh[m], p), "평균예측": p.mean(), "실제평균": yh[m].mean()})
print("\n구간별 확률 보정 확인")
print(pd.DataFrame(ll).round(4).to_string(index=False))

res.to_csv("modelA_segment_results.csv", index=False)
print("저장 완료: pred_valid_modelA.parquet / modelA_segment_results.csv")

/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid val

스태킹 계수: {'GaussianNB': np.float64(-0.01), 'AdaBoost': np.float64(-0.187), 'LightGBM_v2': np.float64(0.769), 'CatBoost_v2': np.float64(0.406)} 절편: 0.025
구간별 규모와 AUC
             구간     행수  사용자수  target평균  AUC_모델A  모델A_CI하한  모델A_CI상한  AUC_LightGBM_v2  AUC_인기도  AUC_사용자기준선
             전체 732898 10929    0.4269   0.6655    0.6615    0.6686           0.6644   0.5626      0.6110
            인기곡 532759 10744    0.4500   0.6598    0.6556    0.6644           0.6592   0.5575      0.6101
           비인기곡 200139  9046    0.3654   0.6604    0.6530    0.6691           0.6591   0.4650      0.6034
         새 아티스트 416707 10623    0.3945   0.6560    0.6508    0.6611           0.6549   0.5676      0.6005
        기존 아티스트 316191  9365    0.4696   0.6663    0.6606    0.6719           0.6653   0.5487      0.6204
        취향 밖 장르  85900  7284    0.3892   0.6542    0.6419    0.6649           0.6533   0.5919      0.5470
        취향 안 장르 646998  9849    0.4319   0.6663    0.6608    0.6706           0.6654   0.5584 

* 4개(NB 포함)와 3개(NB 제외) 스태킹을 같은 조건에서 비교

In [19]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

SEED = 42
N_BOOT = 300
FULL = ["GaussianNB", "AdaBoost", "LightGBM_v2", "CatBoost_v2"]
NO_NB = ["AdaBoost", "LightGBM_v2", "CatBoost_v2"]
SIMPLE = ["LightGBM_v2", "CatBoost_v2"]

va = pd.read_parquet("featv2_valid.parquet", columns=["msno", "target"])
P = pd.read_parquet("pred_valid_models_v2_13.parquet")

y = va["target"].to_numpy()
users = va["msno"].to_numpy()
ur = va["msno"].unique()
rng = np.random.default_rng(SEED)
tune_users = set(rng.choice(ur, len(ur) // 2, replace=False))
is_tune = va["msno"].isin(tune_users).to_numpy()
is_hold = ~is_tune
yt, yh, uh = y[is_tune], y[is_hold], users[is_hold]

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def stack_pred(members):
    Z = np.column_stack([logit(P[m]) for m in members])
    meta = LogisticRegression(C=1.0, max_iter=200).fit(Z[is_tune], yt)
    return meta.predict_proba(Z[is_hold])[:, 1], meta

def ece(yy, p, n_bins=10):
    b = pd.qcut(p, q=n_bins, duplicates="drop")
    s = pd.DataFrame({"y": yy, "p": p, "b": b}).groupby("b", observed=True).agg(y=("y", "mean"), p=("p", "mean"), n=("y", "size"))
    return float((s["n"] * (s["y"] - s["p"]).abs()).sum() / s["n"].sum())

cands = {}
for name, mem in {"4개(NB 포함)": FULL, "3개(NB 제외)": NO_NB, "2개(LGB+Cat)": SIMPLE}.items():
    p, meta = stack_pred(mem)
    cands[name] = p
cands["LightGBM_v2 단일"] = P["LightGBM_v2"].to_numpy()[is_hold]

rows = []
for n, p in cands.items():
    rows.append({"모델": n, "holdout_AUC": roc_auc_score(yh, p), "logloss": log_loss(yh, np.clip(p, 1e-6, 1 - 1e-6)), "ECE": ece(yh, p)})
print(pd.DataFrame(rows).round(5).to_string(index=False))

codes, uniq = pd.factorize(uh)
nu = len(uniq)
grp, kk = {}, {}
for n, p in cands.items():
    _, g = np.unique(p, return_inverse=True)
    grp[n], kk[n] = g, g.max() + 1
yf = yh.astype(float)

def wauc(n, w):
    pos = np.bincount(grp[n], weights=w * yf, minlength=kk[n])
    neg = np.bincount(grp[n], weights=w * (1 - yf), minlength=kk[n])
    below = np.cumsum(neg) - neg
    return float((pos * (below + 0.5 * neg)).sum() / (pos.sum() * neg.sum()))

r = np.random.default_rng(SEED)
boot = {n: np.empty(N_BOOT) for n in cands}
for i in range(N_BOOT):
    cnt = np.bincount(r.integers(0, nu, nu), minlength=nu).astype(float)
    w = cnt[codes]
    for n in cands:
        boot[n][i] = wauc(n, w)

ones = np.ones(len(yf))
pairs = [("4개(NB 포함)", "3개(NB 제외)"), ("4개(NB 포함)", "2개(LGB+Cat)"), ("3개(NB 제외)", "2개(LGB+Cat)"), ("3개(NB 제외)", "LightGBM_v2 단일")]
out = []
for a, b in pairs:
    d = boot[a] - boot[b]
    lo, hi = np.percentile(d, [2.5, 97.5])
    out.append({"비교": f"{a} - {b}", "AUC차이": wauc(a, ones) - wauc(b, ones), "CI하한": lo, "CI상한": hi})
print(pd.DataFrame(out).round(5).to_string(index=False))

_, m4 = stack_pred(FULL)
_, m3 = stack_pred(NO_NB)
print("4개 계수:", dict(zip(FULL, m4.coef_[0].round(3))))
print("3개 계수:", dict(zip(NO_NB, m3.coef_[0].round(3))))

/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid val

            모델  holdout_AUC  logloss     ECE
     4개(NB 포함)      0.66548  0.63975 0.00368
     3개(NB 제외)      0.66515  0.63994 0.00384
   2개(LGB+Cat)      0.66472  0.64017 0.00347
LightGBM_v2 단일      0.66443  0.64065 0.01062
                        비교   AUC차이    CI하한    CI상한
     4개(NB 포함) - 3개(NB 제외) 0.00033 0.00005 0.00066
   4개(NB 포함) - 2개(LGB+Cat) 0.00077 0.00029 0.00118
   3개(NB 제외) - 2개(LGB+Cat) 0.00043 0.00012 0.00073
3개(NB 제외) - LightGBM_v2 단일 0.00072 0.00029 0.00106


/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid val

4개 계수: {'GaussianNB': np.float64(-0.01), 'AdaBoost': np.float64(-0.187), 'LightGBM_v2': np.float64(0.769), 'CatBoost_v2': np.float64(0.406)}
3개 계수: {'AdaBoost': np.float64(-0.215), 'LightGBM_v2': np.float64(0.73), 'CatBoost_v2': np.float64(0.413)}


/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yoonha/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


6. 알고리즘 선정과 합격선 검증

In [20]:
import json
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 42
N_BOOT = 500
AUC_TOL = 0.0005
ECE_MAX = 0.01

CANDIDATES = {
    "스태킹4(NB+Ada+LGB_v2+Cat_v2)": ["GaussianNB", "AdaBoost", "LightGBM_v2", "CatBoost_v2"],
    "스태킹3(Ada+LGB_v2+Cat_v2)": ["AdaBoost", "LightGBM_v2", "CatBoost_v2"],
    "스태킹2(LGB_v2+Cat_v2)": ["LightGBM_v2", "CatBoost_v2"],
    "LightGBM_v2 단일": ["LightGBM_v2"],
}

va = pd.read_parquet("featv2_valid.parquet", columns=["row_order", "msno", "song_id", "target", "is_popular", "is_new_artist", "is_out_of_taste_genre"])
base = pd.read_parquet("baseline_valid_pred.parquet")
P = pd.read_parquet("pred_valid_models_v2_13.parquet")

y = va["target"].to_numpy()
users_all = va["msno"].unique()
rng = np.random.default_rng(SEED)
tune_users = set(rng.choice(users_all, len(users_all) // 2, replace=False))
is_tune = va["msno"].isin(tune_users).to_numpy()
is_hold = ~is_tune
yt, yh = y[is_tune], y[is_hold]
user_h = va["msno"].to_numpy()[is_hold]
pop_h = base["p_pop"].to_numpy()[is_hold]
usr_h = base["p_user_g0"].to_numpy()[is_hold]

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def ece(yy, p, n_bins=10):
    b = pd.qcut(p, q=n_bins, duplicates="drop")
    s = pd.DataFrame({"y": yy, "p": p, "b": b}).groupby("b", observed=True).agg(y=("y", "mean"), p=("p", "mean"), n=("y", "size"))
    return float((s["n"] * (s["y"] - s["p"]).abs()).sum() / s["n"].sum())

def bootstrap_auc_diff(yy, user, p_model, p_base, n_boot=N_BOOT, seed=SEED):
    yy = np.asarray(yy).astype(float)
    codes, uniques = pd.factorize(np.asarray(user))
    n_users = len(uniques)
    _, g_m = np.unique(np.asarray(p_model, dtype=float), return_inverse=True)
    _, g_b = np.unique(np.asarray(p_base, dtype=float), return_inverse=True)
    k_m, k_b = g_m.max() + 1, g_b.max() + 1

    def wauc(g, k, w):
        pos = np.bincount(g, weights=w * yy, minlength=k)
        neg = np.bincount(g, weights=w * (1 - yy), minlength=k)
        below = np.cumsum(neg) - neg
        return float((pos * (below + 0.5 * neg)).sum() / (pos.sum() * neg.sum()))

    ones = np.ones(len(yy))
    auc_m, auc_b = wauc(g_m, k_m, ones), wauc(g_b, k_b, ones)
    r = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        cnt = np.bincount(r.integers(0, n_users, n_users), minlength=n_users).astype(float)
        w = cnt[codes]
        diffs[i] = wauc(g_m, k_m, w) - wauc(g_b, k_b, w)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return auc_m, auc_b, auc_m - auc_b, float(lo), float(hi)

metas, preds_all, preds_h = {}, {}, {}
for name, members in CANDIDATES.items():
    if len(members) == 1:
        preds_all[name] = P[members[0]].to_numpy(dtype=float)
        metas[name] = None
    else:
        Z = np.column_stack([logit(P[m]) for m in members])
        meta = LogisticRegression(C=1.0, max_iter=200).fit(Z[is_tune], yt)
        preds_all[name] = meta.predict_proba(Z)[:, 1]
        metas[name] = meta
    preds_h[name] = preds_all[name][is_hold]

rows = []
for name, members in CANDIDATES.items():
    p = preds_h[name]
    rows.append({"후보": name, "모델수": len(members), "holdout_AUC": roc_auc_score(yh, p),
                 "logloss": log_loss(yh, np.clip(p, 1e-6, 1 - 1e-6)), "ECE": ece(yh, p)})
sel = pd.DataFrame(rows)
print("알고리즘 선정 표 (holdout)")
print(sel.round(5).to_string(index=False))

best_auc = sel["holdout_AUC"].max()
ok = sel[(sel["holdout_AUC"] >= best_auc - AUC_TOL) & (sel["ECE"] <= ECE_MAX)]
chosen = ok.sort_values(["모델수", "holdout_AUC"], ascending=[True, False]).iloc[0]["후보"]
print(f"\n선정 규칙: 최고 AUC와 {AUC_TOL} 이내이고 ECE {ECE_MAX} 이하인 후보 중 모델 수가 가장 적은 것")
print("선정된 알고리즘:", chosen)

p_h = preds_h[chosen]
rows = []
for label, pb in [("인기도 기준선", pop_h), ("사용자 재청취율 기준선", usr_h)]:
    am, ab, d, lo, hi = bootstrap_auc_diff(yh, user_h, p_h, pb)
    rows.append({"비교대상": label, "모델AUC": am, "기준AUC": ab, "AUC차이": d, "CI하한": lo, "CI상한": hi, "합격": bool(d > 0 and lo > 0)})
chk = pd.DataFrame(rows)
print("\n합격선 검증 (사용자 단위 부트스트랩, 95% 신뢰구간)")
print(chk.round(4).to_string(index=False))
passed = bool(chk.iloc[0]["합격"])
print("\n공식 합격선(인기도 대비):", "합격" if passed else "미충족")

h = va[is_hold].reset_index(drop=True)
seg_rows = []
for sname, mask in {
    "전체": np.ones(len(h), dtype=bool),
    "비인기곡": h["is_popular"].to_numpy() == 0,
    "새 아티스트": h["is_new_artist"].to_numpy() == 1,
    "취향 밖 장르": h["is_out_of_taste_genre"].to_numpy() == 1,
}.items():
    am, ab, d, lo, hi = bootstrap_auc_diff(yh[mask], user_h[mask], p_h[mask], usr_h[mask], n_boot=200)
    seg_rows.append({"구간": sname, "행수": int(mask.sum()), "모델AUC": am, "사용자기준선AUC": ab, "AUC차이": d, "CI하한": lo, "CI상한": hi})
print("\n구간별 사용자 기준선 대비 (참고)")
print(pd.DataFrame(seg_rows).round(4).to_string(index=False))

members = CANDIDATES[chosen]
result = {
    "선정_알고리즘": chosen,
    "구성_모델": members,
    "스태킹_계수": None if metas[chosen] is None else dict(zip(members, metas[chosen].coef_[0].round(4).tolist())),
    "스태킹_절편": None if metas[chosen] is None else round(float(metas[chosen].intercept_[0]), 4),
    "선정_규칙": f"최고 AUC와 {AUC_TOL} 이내, ECE {ECE_MAX} 이하 중 모델 수 최소",
    "holdout_AUC": float(sel.loc[sel["후보"] == chosen, "holdout_AUC"].iloc[0]),
    "holdout_ECE": float(sel.loc[sel["후보"] == chosen, "ECE"].iloc[0]),
    "인기도대비_AUC차이": float(chk.iloc[0]["AUC차이"]),
    "인기도대비_CI": [float(chk.iloc[0]["CI하한"]), float(chk.iloc[0]["CI상한"])],
    "합격": passed,
    "부트스트랩": {"단위": "사용자(msno)", "반복수": N_BOOT, "seed": SEED},
}
with open("modelA_선정_합격선_결과.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

joblib.dump({"name": chosen, "members": members, "meta": metas[chosen]}, "modelA_meta.joblib")

out = va[["row_order", "msno", "song_id", "target"]].copy()
out["p_modelA"] = preds_all[chosen]
out["is_tune"] = is_tune
out.to_parquet("pred_valid_modelA_final.parquet", index=False)

sel.to_csv("modelA_후보비교_valid.csv", index=False)
chk.to_csv("modelA_합격선검증_valid.csv", index=False)
print("\n저장 완료: modelA_선정_합격선_결과.json / modelA_meta.joblib / pred_valid_modelA_final.parquet / modelA_후보비교_valid.csv / modelA_합격선검증_valid.csv")

알고리즘 선정 표 (holdout)
                        후보  모델수  holdout_AUC  logloss     ECE
스태킹4(NB+Ada+LGB_v2+Cat_v2)    4      0.66548  0.63975 0.00368
   스태킹3(Ada+LGB_v2+Cat_v2)    3      0.66515  0.63994 0.00384
       스태킹2(LGB_v2+Cat_v2)    2      0.66472  0.64017 0.00347
            LightGBM_v2 단일    1      0.66443  0.64065 0.01062

선정 규칙: 최고 AUC와 0.0005 이내이고 ECE 0.01 이하인 후보 중 모델 수가 가장 적은 것
선정된 알고리즘: 스태킹3(Ada+LGB_v2+Cat_v2)

합격선 검증 (사용자 단위 부트스트랩, 95% 신뢰구간)
        비교대상  모델AUC  기준AUC  AUC차이   CI하한   CI상한   합격
     인기도 기준선 0.6651 0.5626 0.1025 0.0978 0.1080 True
사용자 재청취율 기준선 0.6651 0.6110 0.0542 0.0508 0.0577 True

공식 합격선(인기도 대비): 합격

구간별 사용자 기준선 대비 (참고)
     구간     행수  모델AUC  사용자기준선AUC  AUC차이   CI하한   CI상한
     전체 732898 0.6651     0.6110 0.0542 0.0512 0.0576
   비인기곡 200139 0.6603     0.6034 0.0569 0.0506 0.0624
 새 아티스트 416707 0.6554     0.6005 0.0549 0.0507 0.0593
취향 밖 장르  85900 0.6533     0.5470 0.1063 0.0912 0.1196

저장 완료: modelA_선정_합격선_결과.json / modelA_meta.joblib / pred_valid_modelA_fi

7. 최종

In [21]:
import json
import os
import time
import warnings
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 42
SAMPLE_N = 500000
SAMPLE_SLOW = 200000
N_BOOT = 500
CAT = "primary_genre_code"
KEYS = {"row_order", "msno", "song_id", "target"}
DROP_NEW = {"user_repeat_song_ratio"}
DONE_FLAG = "test_eval_done.json"

if os.path.exists(DONE_FLAG):
    raise SystemExit("test 평가는 이미 한 번 수행했습니다. 결과를 보고 모델을 바꾸지 않기 위해 중단합니다. 다시 실행하려면 test_eval_done.json을 삭제하세요.")

t_all = time.time()
tr_all = pd.concat([pd.read_parquet("featv2_train_a.parquet"), pd.read_parquet("featv2_train_b.parquet")], ignore_index=True)
tr = tr_all.sample(n=SAMPLE_N, random_state=SEED).reset_index(drop=True)
del tr_all
va = pd.read_parquet("featv2_valid.parquet")
te = pd.read_parquet("featv2_test.parquet")
trb = pd.read_parquet("featv2_train_b.parquet", columns=["target", "song_cnt"])

FEATS = [c for c in va.columns if c not in KEYS and not c.endswith("_g30") and c not in DROP_NEW]
print("사용 특성 수:", len(FEATS))

users = va["msno"].unique()
rng = np.random.default_rng(SEED)
tune_users = set(rng.choice(users, len(users) // 2, replace=False))
is_tune = va["msno"].isin(tune_users).to_numpy()
is_hold = ~is_tune
yv = va["target"].to_numpy()
yt, yh = yv[is_tune], yv[is_hold]
ytr = tr["target"].to_numpy()

def prep(d, cols=None):
    X = d[FEATS].copy() if cols is None else cols.copy()
    X[CAT] = X[CAT].astype(int) + 1
    return X

Xtr, Xva = prep(tr), prep(va)
Xt = Xva[is_tune]
sub = np.random.default_rng(SEED).choice(len(tr), SAMPLE_SLOW, replace=False)

ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=4, min_samples_leaf=200), n_estimators=60, learning_rate=0.5, random_state=SEED)
ada.fit(Xtr.fillna(-1).iloc[sub], ytr[sub])

lgbm = lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.03, num_leaves=31, max_depth=6, min_child_samples=500, subsample=0.7, subsample_freq=1, colsample_bytree=0.6, reg_lambda=10.0, n_jobs=-1, random_state=SEED + 1, verbose=-1)
lgbm.fit(Xtr, ytr, eval_set=[(Xt, yt)], eval_metric="auc", categorical_feature=[CAT], callbacks=[lgb.early_stopping(40, verbose=False)])

cat = CatBoostClassifier(iterations=1500, learning_rate=0.06, depth=5, l2_leaf_reg=10, random_seed=SEED + 1, verbose=0, thread_count=-1, cat_features=[CAT], eval_metric="AUC", od_type="Iter", od_wait=40, use_best_model=True)
cat.fit(Xtr, ytr, eval_set=(Xt, yt))
print(f"학습 완료 {time.time() - t_all:.0f}초 | LightGBM 반복 {lgbm.best_iteration_}, CatBoost 반복 {cat.get_best_iteration()}")

saved = joblib.load("modelA_meta.joblib")
members, meta = saved["members"], saved["meta"]
print("모델 A 구성:", members)

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def predict_A(X):
    parts = {
        "AdaBoost": ada.predict_proba(X.fillna(-1))[:, 1],
        "LightGBM_v2": lgbm.predict_proba(X, num_iteration=lgbm.best_iteration_)[:, 1],
        "CatBoost_v2": cat.predict_proba(X)[:, 1],
    }
    Z = np.column_stack([logit(parts[m]) for m in members])
    return meta.predict_proba(Z)[:, 1]

p_va = predict_A(Xva)
print(f"[점검] valid holdout AUC {roc_auc_score(yh, p_va[is_hold]):.5f} (6단계 선정 시 0.66515와 비슷해야 함)")

Xte0 = prep(te)
Xte30 = Xte0.copy()
g_cols = [c for c in FEATS if c.endswith("_g0")]
for c in g_cols:
    Xte30[c] = te[c[:-3] + "_g30"].to_numpy()
print("g30으로 대체한 열 수:", len(g_cols), "| g30 결측률:", round(float(Xte30[g_cols].isna().mean().mean()), 4))

yte = te["target"].to_numpy()
p_g0 = predict_A(Xte0)
p_g30 = predict_A(Xte30)

pop_lr = LogisticRegression().fit(np.log1p(trb[["song_cnt"]].to_numpy()), trb["target"])
p_pop = pop_lr.predict_proba(np.log1p(te[["song_cnt"]].to_numpy()))[:, 1]
p_usr = te["user_rate_g0"].to_numpy(dtype=float)
p_usr = np.where(np.isnan(p_usr), np.nanmean(p_usr), p_usr)

def ece(yy, p, n_bins=10):
    b = pd.qcut(p, q=n_bins, duplicates="drop")
    s = pd.DataFrame({"y": yy, "p": p, "b": b}).groupby("b", observed=True).agg(y=("y", "mean"), p=("p", "mean"), n=("y", "size"))
    return float((s["n"] * (s["y"] - s["p"]).abs()).sum() / s["n"].sum())

def boot_diff(yy, user, p_m, p_b, n_boot=N_BOOT):
    yy = np.asarray(yy).astype(float)
    codes, uniq = pd.factorize(np.asarray(user))
    nu = len(uniq)
    _, gm = np.unique(np.asarray(p_m, dtype=float), return_inverse=True)
    _, gb = np.unique(np.asarray(p_b, dtype=float), return_inverse=True)
    km, kb = gm.max() + 1, gb.max() + 1

    def wauc(g, k, w):
        pos = np.bincount(g, weights=w * yy, minlength=k)
        neg = np.bincount(g, weights=w * (1 - yy), minlength=k)
        below = np.cumsum(neg) - neg
        return float((pos * (below + 0.5 * neg)).sum() / (pos.sum() * neg.sum()))

    ones = np.ones(len(yy))
    am, ab = wauc(gm, km, ones), wauc(gb, kb, ones)
    r = np.random.default_rng(SEED)
    d = np.empty(n_boot)
    for i in range(n_boot):
        w = np.bincount(r.integers(0, nu, nu), minlength=nu).astype(float)[codes]
        d[i] = wauc(gm, km, w) - wauc(gb, kb, w)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return am, ab, am - ab, float(lo), float(hi)

ut = te["msno"].to_numpy()
rows = []
for name, p in [("모델A(g0)", p_g0), ("모델A(g30)", p_g30), ("인기도 기준선", p_pop), ("사용자 재청취율 기준선(g0)", p_usr)]:
    pc = np.clip(p, 1e-6, 1 - 1e-6)
    rows.append({"모델": name, "AUC": roc_auc_score(yte, p), "logloss": log_loss(yte, pc), "ECE": ece(yte, p), "평균예측": float(p.mean()), "실제평균": float(yte.mean())})
print("\ntest 성능 (1회 평가)")
print(pd.DataFrame(rows).round(4).to_string(index=False))

chk = []
for label, pm in [("g0", p_g0), ("g30", p_g30)]:
    for bl, pb in [("인기도", p_pop), ("사용자 기준선", p_usr)]:
        am, ab, d, lo, hi = boot_diff(yte, ut, pm, pb)
        chk.append({"모델": f"모델A({label})", "비교대상": bl, "모델AUC": am, "기준AUC": ab, "AUC차이": d, "CI하한": lo, "CI상한": hi, "합격": bool(d > 0 and lo > 0)})
chk = pd.DataFrame(chk)
print("\ntest 합격선 검증 (사용자 단위 부트스트랩)")
print(chk.round(4).to_string(index=False))

segs = {
    "전체": np.ones(len(te), dtype=bool),
    "인기곡": te["is_popular"].to_numpy() == 1,
    "비인기곡": te["is_popular"].to_numpy() == 0,
    "새 아티스트": te["is_new_artist"].to_numpy() == 1,
    "취향 밖 장르": te["is_out_of_taste_genre"].to_numpy() == 1,
    "비인기곡+새 아티스트": (te["is_popular"].to_numpy() == 0) & (te["is_new_artist"].to_numpy() == 1),
}
seg_rows = []
for sname, m in segs.items():
    if m.sum() > 1000 and 0 < yte[m].mean() < 1:
        am, ab, d, lo, hi = boot_diff(yte[m], ut[m], p_g0[m], p_usr[m], n_boot=200)
        seg_rows.append({"구간": sname, "행수": int(m.sum()), "AUC_g0": am, "AUC_g30": roc_auc_score(yte[m], p_g30[m]), "사용자기준선AUC": ab, "차이(g0)": d, "CI하한": lo, "CI상한": hi})
print("\ntest 구간별 성능")
print(pd.DataFrame(seg_rows).round(4).to_string(index=False))

out = te[["row_order", "msno", "song_id", "target"]].copy()
out["p_g0"] = p_g0
out["p_g30"] = p_g30
out["rank_g0"] = out.groupby("msno")["p_g0"].rank(method="first", ascending=False)
top10 = out[out["rank_g0"] <= 10].sort_values(["msno", "rank_g0"])
ucnt = out.groupby("msno").size()
elig = ucnt[ucnt >= 10].index
top_mean = top10[top10["msno"].isin(elig)]["target"].mean()
all_mean = out[out["msno"].isin(elig)]["target"].mean()
print(f"\n후보 10곡 이상 사용자 {len(elig)}명 | 상위 10곡 재청취율 {top_mean:.4f} vs 전체 후보 {all_mean:.4f}")

out.to_parquet("pred_test_modelA.parquet", index=False)
top10.to_csv("recommend_top10_test_modelA.csv", index=False)
joblib.dump({"ada": ada, "lgbm": lgbm}, "modelA_sk_lgbm.joblib")
cat.save_model("modelA_catboost.cbm")
pd.DataFrame(rows).to_csv("modelA_test_성능.csv", index=False)
chk.to_csv("modelA_test_합격선검증.csv", index=False)
pd.DataFrame(seg_rows).to_csv("modelA_test_구간별.csv", index=False)
with open("modelA_특성목록.json", "w", encoding="utf-8") as f:
    json.dump({"features": FEATS, "categorical": CAT, "g30_대체열": g_cols}, f, ensure_ascii=False, indent=2)
with open(DONE_FLAG, "w", encoding="utf-8") as f:
    json.dump({"평가": "완료", "g0_AUC": float(roc_auc_score(yte, p_g0)), "g30_AUC": float(roc_auc_score(yte, p_g30))}, f)
print("저장 완료")

사용 특성 수: 45
학습 완료 87초 | LightGBM 반복 477, CatBoost 반복 490
모델 A 구성: ['AdaBoost', 'LightGBM_v2', 'CatBoost_v2']
[점검] valid holdout AUC 0.66515 (6단계 선정 시 0.66515와 비슷해야 함)
g30으로 대체한 열 수: 9 | g30 결측률: 0.0

test 성능 (1회 평가)
              모델    AUC  logloss    ECE   평균예측   실제평균
         모델A(g0) 0.6528   0.6397 0.0145 0.3900 0.4045
        모델A(g30) 0.6199   0.6682 0.0735 0.4779 0.4045
         인기도 기준선 0.5501   0.6858 0.0898 0.4798 0.4045
사용자 재청취율 기준선(g0) 0.6052   0.7063 0.1189 0.5179 0.4045

test 합격선 검증 (사용자 단위 부트스트랩)
      모델    비교대상  모델AUC  기준AUC  AUC차이   CI하한   CI상한   합격
 모델A(g0)     인기도 0.6528 0.5501 0.1027 0.0984 0.1077 True
 모델A(g0) 사용자 기준선 0.6528 0.6052 0.0476 0.0447 0.0504 True
모델A(g30)     인기도 0.6199 0.5501 0.0698 0.0656 0.0741 True
모델A(g30) 사용자 기준선 0.6199 0.6052 0.0147 0.0111 0.0181 True

test 구간별 성능
         구간      행수  AUC_g0  AUC_g30  사용자기준선AUC  차이(g0)   CI하한   CI상한
         전체 1471022  0.6528   0.6199     0.6052  0.0476 0.0451 0.0504
        인기곡 1054451  0.6432   0.6057     0.6046 

8. 요약

In [22]:
import json
import os
import shutil
from datetime import datetime
import pandas as pd

OUT = "modelA_deliverables"
os.makedirs(OUT, exist_ok=True)

FILES = {
    "예측": [
        "pred_valid_modelA_final.parquet",
        "pred_test_modelA.parquet",
        "recommend_top10_test_modelA.csv",
    ],
    "모델": [
        "modelA_meta.joblib",
        "modelA_sk_lgbm.joblib",
        "modelA_catboost.cbm",
    ],
    "특성": [
        "modelA_특성목록.json",
        "featv2_train_a.parquet",
        "featv2_train_b.parquet",
        "featv2_valid.parquet",
        "featv2_test.parquet",
    ],
    "검증결과": [
        "합격선_기준.json",
        "합격선_기준.md",
        "modelA_선정_합격선_결과.json",
        "modelA_후보비교_valid.csv",
        "modelA_합격선검증_valid.csv",
        "modelA_test_성능.csv",
        "modelA_test_합격선검증.csv",
        "modelA_test_구간별.csv",
        "modelA_segment_results.csv",
        "ensemble_combo_results_v2_13.csv",
        "test_eval_done.json",
    ],
}
COPY_FEATURES = False

rows = []
for kind, names in FILES.items():
    sub = os.path.join(OUT, kind)
    os.makedirs(sub, exist_ok=True)
    for n in names:
        if kind == "특성" and n.startswith("featv2_") and not COPY_FEATURES:
            rows.append({"구분": kind, "파일": n, "상태": "원본 위치 유지", "크기MB": round(os.path.getsize(n) / 1e6, 1) if os.path.exists(n) else None})
            continue
        if os.path.exists(n):
            shutil.copy2(n, os.path.join(sub, n))
            rows.append({"구분": kind, "파일": n, "상태": "복사", "크기MB": round(os.path.getsize(n) / 1e6, 2)})
        else:
            rows.append({"구분": kind, "파일": n, "상태": "없음", "크기MB": None})
manifest = pd.DataFrame(rows)
manifest.to_csv(os.path.join(OUT, "파일목록.csv"), index=False, encoding="utf-8-sig")
print(manifest.to_string(index=False))

def load_json(p):
    return json.load(open(p, encoding="utf-8")) if os.path.exists(p) else {}

sel = load_json("modelA_선정_합격선_결과.json")
feat = load_json("modelA_특성목록.json")
crit = load_json("합격선_기준.json")
test_perf = pd.read_csv("modelA_test_성능.csv") if os.path.exists("modelA_test_성능.csv") else pd.DataFrame()
test_chk = pd.read_csv("modelA_test_합격선검증.csv") if os.path.exists("modelA_test_합격선검증.csv") else pd.DataFrame()
test_seg = pd.read_csv("modelA_test_구간별.csv") if os.path.exists("modelA_test_구간별.csv") else pd.DataFrame()
val_cmp = pd.read_csv("modelA_후보비교_valid.csv") if os.path.exists("modelA_후보비교_valid.csv") else pd.DataFrame()

def md(df):
    return df.round(4).to_markdown(index=False) if len(df) else "(파일 없음)"

try:
    import tabulate
except ImportError:
    def md(df):
        return "```\n" + df.round(4).to_string(index=False) + "\n```" if len(df) else "(파일 없음)"

lines = []
lines.append("# 모델 A 산출물 요약")
lines.append(f"\n작성: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
lines.append("\n## 1. 모델 A 정의")
lines.append("KKBox 데이터로 학습한 재청취 확률 예측 모델입니다. 다양성 보정(B1·B2)은 적용하지 않은 기본 개인화 모델입니다.")
lines.append(f"- 구성: {', '.join(sel.get('구성_모델', []))} 의 확률(logit)을 로지스틱 회귀로 결합(스태킹)")
lines.append(f"- 스태킹 계수: {sel.get('스태킹_계수')}, 절편 {sel.get('스태킹_절편')}")
lines.append(f"- 선정 규칙: {sel.get('선정_규칙')}")
lines.append(f"- 학습 데이터: train_a + train_b에서 무작위 50만 행(AdaBoost는 그중 20만 행), 조기 종료는 valid 사용자 절반")
lines.append(f"- 입력 특성 {len(feat.get('features', []))}개(`_g0` 계열 포함, `_g30`·`source_*`·키 열 제외)")
lines.append("\n## 2. 데이터 분할")
lines.append("`row_order / 7,377,418` 비율 기준 블록 분할: 준비 기간(0~0.2, 기록 전용), train_a(0.2~0.4), train_b(0.4~0.6), valid(0.6~0.8), test(0.8~1.0). 모든 특성은 블록 시작 시점 이전 기록으로만 계산하고 `_g0`(지연 없음)·`_g30`(30일 확정분) 두 버전을 만들었습니다.")
lines.append("\n## 3. 사전 합격선")
lines.append(f"{crit.get('합격선', '')}")
lines.append("\n## 4. valid 후보 비교 (holdout)")
lines.append(md(val_cmp))
lines.append("\n## 5. test 최종 평가 (1회)")
lines.append(md(test_perf))
lines.append("\n### 합격선 검증")
lines.append(md(test_chk))
lines.append("\n### 구간별 성능")
lines.append(md(test_seg))
lines.append("\n## 6. 한계와 미해결 항목")
lines.append("- `_g30`(현실 조건) 평가에서 AUC가 크게 낮아지고 확률이 과대예측됩니다. 서비스 적용 시 `_g30` 조건 학습 또는 확률 보정이 필요합니다.")
lines.append("- `membership_days`는 만료일을 사용해 미래 정보일 수 있습니다. 팀 확인이 필요합니다.")
lines.append("- 후보곡이 실제로 들은 곡뿐이라 선택 편향이 있습니다. 상위 10곡 재청취율은 절대값이 아니라 순위 정렬 확인용입니다.")
lines.append("- `valid`를 여러 번 보고 설계를 정했으므로 valid 점수는 낙관적일 수 있고, test는 1회만 평가했습니다. 이후 모델·특성 변경은 사후 조정입니다.")
lines.append("- `pred_valid_modelA_final.parquet`의 tune 절반은 메타 모델 학습 구간(in-sample)이라 `is_tune` 열로 구분해 사용합니다.")
lines.append("- 시간에 따른 target 평균 하락(0.548 → 0.405)으로 test AUC가 valid보다 낮습니다.")
lines.append("- Kaggle 데이터 이용 조건과 나이·성별 사용 가능 여부는 외부 공개 전 확인이 필요합니다.")
lines.append("\n## 7. 다음 단계")
lines.append("로드맵 8단계(B1 인기도 보정, B2 취향 확장 보정, 정책 1~3 설계)로 이동합니다. B1·B2는 `pred_valid_modelA_final.parquet`의 holdout 절반과 `pred_test_modelA.parquet`의 확률을 재정렬하는 방식으로 구현합니다.")
lines.append("\n## 8. 파일 구성")
lines.append(md(manifest))

with open(os.path.join(OUT, "README_modelA.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

check = pd.read_parquet("pred_test_modelA.parquet", columns=["msno", "song_id", "p_g0", "p_g30"])
print("\n점검: test 예측 행", len(check), "| 사용자", check["msno"].nunique(), "| p_g0 범위", round(float(check["p_g0"].min()), 3), "~", round(float(check["p_g0"].max()), 3))
print("저장 완료:", OUT)

  구분                               파일       상태   크기MB
  예측  pred_valid_modelA_final.parquet       복사  79.83
  예측         pred_test_modelA.parquet       복사  93.46
  예측  recommend_top10_test_modelA.csv       복사  28.51
  모델               modelA_meta.joblib       복사   0.00
  모델            modelA_sk_lgbm.joblib       복사   1.78
  모델              modelA_catboost.cbm       복사   0.43
  특성                 modelA_특성목록.json       복사   0.00
  특성           featv2_train_a.parquet 원본 위치 유지 124.80
  특성           featv2_train_b.parquet 원본 위치 유지 129.70
  특성             featv2_valid.parquet 원본 위치 유지 143.60
  특성              featv2_test.parquet 원본 위치 유지 149.90
검증결과                      합격선_기준.json       복사   0.00
검증결과                        합격선_기준.md       복사   0.00
검증결과            modelA_선정_합격선_결과.json       복사   0.00
검증결과            modelA_후보비교_valid.csv       복사   0.00
검증결과           modelA_합격선검증_valid.csv       복사   0.00
검증결과               modelA_test_성능.csv       복사   0.00
검증결과            modelA_test_